# 05 — Touch Objects: Coaching Session

**Task**: Reach out and touch a target object on the table. The arm starts at home position, touches the object, and returns.

**Why start here**: Touch is the simplest manipulation task — no grasp required. It builds muscle memory for smooth arm motion and gives the policy a clear contact event to learn from.

**Prereqs**:
- `00_arm_setup` complete — arms calibrated, reference animations recorded
- `01_reserve_node` complete — MI100 provisioned
- `.env` populated
- Target object placed on the table in a fixed, repeatable position

**Outcome**: Dataset pushed to HF Hub, ACT policy trained on MI100, checkpoint available.

---

In [ ]:
import os
import shlex
import time
import json
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

_bench = {"notebook": "05_touch_objects", "started_at": datetime.utcnow().isoformat(),
          "timings": {}, "config": {}}
_t0 = time.monotonic()

PI_HOST      = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT      = os.getenv("PI_PORT", "22222")
PI_USER      = "root"
HF_USER      = os.getenv("HF_USER")
HF_TOKEN     = os.getenv("HF_TOKEN")
FLOATING_IP  = os.getenv("CONTROL_FLOATING_IP")
PI_IMAGE     = f"{HF_USER}/lerobot-soarm101:latest"
POLICY       = os.getenv("POLICY", "act")

# ── Task configuration ────────────────────────────────────────────────────────
TASK_SLUG    = os.getenv("TASK_TOUCH_SLUG",  "touch-block")
TASK_DESC    = os.getenv("TASK_TOUCH_DESC",  "Touch the red block on the table")
REF_REPO     = f"{HF_USER}/soarm101-{TASK_SLUG}-reference"
DATASET_REPO = f"{HF_USER}/soarm101-{TASK_SLUG}"
MODEL_REPO   = f"{HF_USER}/{POLICY}-{TASK_SLUG}"

# Coaching session parameters
NUM_EPISODES = 50   # collect 50 episodes for touch task
EPISODE_TIME = 20   # touch is a short task — 20s per episode
RESET_TIME   = 8

_bench["config"] = {"task": TASK_SLUG, "dataset": DATASET_REPO,
                    "model": MODEL_REPO, "policy": POLICY}

print(f"Task:    {TASK_DESC}")
print(f"Dataset: {DATASET_REPO}")
print(f"Model:   {MODEL_REPO}")
print(f"Node:    {FLOATING_IP}")
total = NUM_EPISODES * (EPISODE_TIME + RESET_TIME) - RESET_TIME
print(f"Session: {NUM_EPISODES} episodes × {EPISODE_TIME}s  ≈ {total//60}m {total%60}s")

## 1. Replay Reference Animation

Show the student what the task looks like before they start coaching. The follower arm replays an operator-recorded example.

In [ ]:
print(f"Playing reference animation from {REF_REPO}")
print("Watch the follower arm — this is what your demonstrations should look like.")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-replay \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.id=alpha_follower \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={REF_REPO} \\')
print(f'     --dataset.episode=0 \\')
print(f'     --play_sounds=false"')

input("\nPress Enter after watching the reference animation: ")

## 2. Coach the Robot — Touch Task

Use the leader arm to demonstrate the touch task. The follower arm records your motion.

**Coaching tips**:
- Begin and end every episode at the exact same home position
- Move smoothly — hesitations and corrections confuse the policy
- Make contact clearly — press lightly on the object for ~1s
- If an episode goes wrong, press Stop (or Ctrl-C) — it will be discarded

In [ ]:
print(f"Ready to coach: {TASK_DESC}")
print(f"Collecting {NUM_EPISODES} episodes × {EPISODE_TIME}s")
print()
print("Run in your terminal:")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video2 \\')
print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   -e HF_TOKEN={HF_TOKEN} \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-record \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={DATASET_REPO} \\')
print(f'     --dataset.num_episodes={NUM_EPISODES} \\')
print(f"     --dataset.task='{TASK_DESC}' \\'")
print(f'     --dataset.push_to_hub=true"')

input("\nPress Enter when coaching session is complete: ")
print(f"Dataset: https://huggingface.co/datasets/{DATASET_REPO}")

## 3. Verify Episode Quality — Replay Before Training

Replay episode 0 of your coaching data before committing to a full training run.

In [ ]:
print(f"Replaying episode 0 from {DATASET_REPO}")
print("Compare to the reference — does it look like the task?")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-replay \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.id=alpha_follower \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={DATASET_REPO} \\')
print(f'     --dataset.episode=0 \\')
print(f'     --play_sounds=false"')

quality = input("\nData quality OK? [y/N]: ")
if quality.strip().lower() not in ('y', 'yes'):
    print("Re-run coaching cell and collect better episodes before training.")
else:
    print("Quality confirmed — proceeding to training.")

## 4. Train on MI100

In [ ]:
import re
_t_train = time.monotonic()

if not FLOATING_IP or FLOATING_IP == "REPLACE_ME_AFTER_PROVISION":
    raise RuntimeError("CONTROL_FLOATING_IP not set — run 01_reserve_node first")

train_cmd = (
    f"source ~/miniconda3/bin/activate lerobot && "
    f"cd ~/lerobot && "
    f"python lerobot/scripts/train.py "
    f"--dataset.repo_id={DATASET_REPO} "
    f"--policy.path=lerobot/{POLICY} "
    f"--output_dir=outputs/train/{POLICY}_{TASK_SLUG} "
    f"--job_name={POLICY}_{TASK_SLUG} "
    f"--policy.device=cuda "
    f"--wandb.enable=false"
)

launch = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"tmux kill-session -t train 2>/dev/null || true; "
     f"tmux new-session -d -s train '{train_cmd}' && echo launched"],
    capture_output=True, text=True
)
if "launched" in launch.stdout:
    print(f"Training launched on {FLOATING_IP} in tmux session 'train'")
    print(f"  Monitor: ssh cc@{FLOATING_IP}  →  tmux attach -t train")
else:
    print(f"Launch failed: {launch.stderr.strip()}")

_bench["timings"]["training_start_s"] = round(time.monotonic() - _t_train, 1)

## 5. Monitor and Upload

In [ ]:
TOTAL_STEPS   = 80000   # touch task trains faster — fewer steps needed
POLL_INTERVAL = 60
LOG_PATH      = f"~/lerobot/outputs/train/{POLICY}_{TASK_SLUG}/train.log"

with tqdm(total=TOTAL_STEPS, unit="step", desc="Training") as pbar:
    last_step = 0
    for _ in range(90):  # poll up to 90 min
        out = subprocess.run(
            ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
             f"tail -n 3 {LOG_PATH} 2>/dev/null"],
            capture_output=True, text=True
        ).stdout.strip()
        steps = re.findall(r'step=(\d+)', out)
        if steps:
            cur = int(steps[-1])
            pbar.update(cur - last_step)
            last_step = cur
        if out:
            pbar.set_postfix_str(out.splitlines()[-1][:60])
        if last_step >= TOTAL_STEPS:
            break
        time.sleep(POLL_INTERVAL)

# Upload checkpoint
print("\nUploading checkpoint...")
up = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"HF_TOKEN={HF_TOKEN} huggingface-cli upload {MODEL_REPO} "
     f"~/lerobot/outputs/train/{POLICY}_{TASK_SLUG}/checkpoints/last/ ."],
    capture_output=True, text=True
)
if up.returncode == 0:
    print(f"Checkpoint: https://huggingface.co/{MODEL_REPO}")
else:
    print(f"Upload failed: {up.stderr.strip()}")

_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"05_touch_objects_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")